# Bigram / Mini Transformer Language Model

Character-level language model trained on Tiny Shakespeare, built from a bigram baseline up to a small Transformer.

**Improvements vs original:** larger context & model, GELU MLP, weight tying, higher LR + warmup, longer training, temperature/top-k sampling for cleaner generation.

## 1. Setup

In [1]:
import math
import os

import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## 2. Config

In [16]:
block_size = 64        
n_embd = 128         
n_head = 4             
n_layer = 6           
dropout = 0.2          

batch_size = 32
learning_rate = 1e-3   
max_iters = 5000
warmup_iters = 200
eval_interval = 250
eval_iters = 40
grad_clip = 1.0

temperature = 0.8
top_k = 40

checkpoint_path = "../checkpoints/model.pt"

## 3. Load Data

In [4]:
with open("../data/tiny shakespeare.txt", "r") as f:
    text = f.read()

## 4. Build the Character Vocabulary

In [6]:
vocab = sorted(list(set(text)))
vocab_size = len(vocab)
vocab_size

65

## 5. Custom Tokenizer

In [7]:
enc = {ch: i for i, ch in enumerate(vocab)}
dec = {i: ch for i, ch in enumerate(vocab)}

encode = lambda s: [enc[i] for i in s]
decode = lambda l: ''.join([dec[i] for i in l])

print(encode("hello world"))
decode(encode("hello world"))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]


'hello world'

## 6. Train / Test Split

In [8]:
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}")
if data.ndim > 1:
    data = data.flatten()
    print(f"Flattened data shape: {data.shape}")

n = int(0.9 * len(data))
train_set = data[:n]
test_set = data[n:]

Data shape: torch.Size([1115394])


## 7. Batching

Samples a fresh random batch on every call instead of pre-materializing every window in memory.

In [9]:
def get_batch(split):
    source = train_set if split == "train" else test_set
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print(f"xb shape: {xb.shape}")
print(f"yb shape: {yb.shape}")

xb shape: torch.Size([32, 64])
yb shape: torch.Size([32, 64])


## 8. Transformer Building Blocks

In [11]:
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


def top_k_logits(logits, k):
    if k is None or k <= 0:
        return logits
    v, _ = torch.topk(logits, min(k, logits.size(-1)))
    return logits.masked_fill(logits < v[:, [-1]], float("-inf"))

## The Transformer

In [12]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding_table.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            logits = top_k_logits(logits, top_k)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        self.train()
        return idx


model = LanguageModel(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)
xb, yb = get_batch("train")
logits, loss = model(xb, yb)
n_params = sum(p.numel() for p in model.parameters())
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss}")

Logits shape: torch.Size([32, 64, 65])
Loss: 4.204956531524658


## 11. Prediction Before Training

In [13]:
start_context = torch.zeros((1, 1), dtype=torch.long, device=device)

untrained_output = model.generate(
    start_context.clone(), max_new_tokens=300, temperature=1.0, top_k=None
)[0].tolist()
print(decode(untrained_output))


ZPD!U3m'grMbZOafuUBd.qvwfNxOCJzdd'.k g-?-GBF3MBL:HWRirEErV
kI yvL&YLtp-,vFiJQQodhFkWYFAaR-DaHwrtIX3IRLfJYN CLlmfBec&uSNPdlKVAawGE3-HLdruLgODR,EJidp,j
eXoC3SfUPATrU;?&MUYpAhKuJVV;Vukx$?Nbz-BkPhtsChFMMpk-Y?FpMngUf!Zs,fjwo$!AMK.BkBKSuQFw&RUMMyy s,zroS.:QHlESl?.C mmtHIRbm;kwbaqMqC-x  xwz ;3E.vBlhGrGfX:Z


## 12. Training Loop

Loss is averaged over `eval_iters` batches for a stable estimate. Gradient clipping and a cosine learning-rate schedule are applied every step.

In [17]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "test"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    progress = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    return learning_rate * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.99), weight_decay=0.1)

train_losses = []
test_losses = []
best_test = float("inf")

for i in range(max_iters):
    lr = get_lr(i)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    if i % eval_interval == 0 or i == max_iters - 1:
        losses = estimate_loss()
        train_losses.append(losses["train"])
        test_losses.append(losses["test"])
        print(
            f"Iteration {i}: Train Loss {losses['train']:.4f} | "
            f"Test Loss {losses['test']:.4f} | LR {lr:.2e}"
        )
        if losses["test"] < best_test:
            best_test = losses["test"]
            torch.save(
                {
                    "model": model.state_dict(),
                    "config": {
                        "vocab_size": vocab_size,
                        "n_embd": n_embd,
                        "n_head": n_head,
                        "n_layer": n_layer,
                        "block_size": block_size,
                        "dropout": dropout,
                    },
                    "test_loss": best_test,
                },
                checkpoint_path,
            )

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

print(f"Final step loss: {loss.item():.4f}")

Iteration 0: Train Loss 4.2148 | Test Loss 4.2157 | LR 5.00e-06
Iteration 250: Train Loss 2.4490 | Test Loss 2.4555 | LR 1.00e-03
Iteration 500: Train Loss 2.1874 | Test Loss 2.2056 | LR 9.91e-04
Iteration 750: Train Loss 2.0210 | Test Loss 2.0770 | LR 9.71e-04
Iteration 1000: Train Loss 1.9091 | Test Loss 1.9963 | LR 9.40e-04
Iteration 1250: Train Loss 1.8319 | Test Loss 1.9481 | LR 8.98e-04
Iteration 1500: Train Loss 1.7551 | Test Loss 1.9080 | LR 8.47e-04
Iteration 1750: Train Loss 1.7063 | Test Loss 1.8643 | LR 7.88e-04
Iteration 2000: Train Loss 1.6777 | Test Loss 1.8321 | LR 7.22e-04
Iteration 2250: Train Loss 1.6312 | Test Loss 1.8020 | LR 6.52e-04
Iteration 2500: Train Loss 1.5855 | Test Loss 1.7626 | LR 5.79e-04
Iteration 2750: Train Loss 1.5725 | Test Loss 1.7362 | LR 5.06e-04
Iteration 3000: Train Loss 1.5625 | Test Loss 1.7284 | LR 4.34e-04
Iteration 3250: Train Loss 1.5332 | Test Loss 1.6970 | LR 3.64e-04
Iteration 3500: Train Loss 1.4956 | Test Loss 1.6862 | LR 3.00e-04
I

## 13. Plot Loss Curve

In [18]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="train_loss")
plt.plot(test_losses, label="test_loss")
plt.xlabel(f"Evaluation step (every {eval_interval} iterations)")
plt.ylabel("Loss")
plt.title("Training Progress")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=120)
plt.show()
print("Saved loss_curve.png")

Saved loss_curve.png


/tmp/ipykernel_153755/4073015294.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Save & Reload Checkpoint

In [19]:
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
cfg = ckpt["config"]
loaded_model = LanguageModel(
    cfg["vocab_size"], cfg["n_embd"], cfg["n_head"], cfg["n_layer"], cfg["block_size"], cfg["dropout"]
).to(device)
loaded_model.load_state_dict(ckpt["model"])
loaded_model.eval()
print(f"Loaded best checkpoint (test loss={ckpt['test_loss']:.4f})")

Loaded best checkpoint (test loss=1.6245)


## 15. Prediction After Training

In [20]:
print("--- AFTER TRAINING (temperature/top-k sampling) ---")
torch.manual_seed(42)
trained_output = loaded_model.generate(
    start_context.clone(),
    max_new_tokens=1000,
    temperature=temperature,
    top_k=top_k,
)[0].tolist()
print(decode(trained_output))

print("\n=== COMPARISON ===")
print(f"Improved test loss : {ckpt['test_loss']:.4f}")
print(f"Improved params    : {n_params / 1e3:.1f}K")
print(f"Context length     : 32 -> {block_size}")
print(f"Embedding / layers : 64x4 -> {n_embd}x{n_layer}")

--- AFTER TRAINING (temperature/top-k sampling) ---


KING RICHARD III:
God you will stood and be counter eyes and desire,
The report we made the no heart, that is u
cousin the proceed of his crulence, and my lord?

HENRY BOLINGBROKE:
That hence you, but me be die you us and woman's life?
When are thou do forth a not so
she comes; come I should all be petition,
The services to him.

First Citizen:
So, thou shalt I she wringth,-he was I will fare a
in Claudious up to maid, his are is come in my deed:
To an for thy death duke brother's true, and home
From cured hath rightly like party have made men
These that morning traitors, it, thou sea so sking of
To our feell, or madam, it shall shall not we have danger.

CORIOLANUS:
Sir, ere my lords, to the lord, I shall be busia's wealt
To and the father sight tother gold when your words.

ISABELLA:
O know you, and all but and for what show my son?
I fair happined heart, we are love may which you
deserval and these from every woman: to me my lord